In [4]:
# Project Name: CampusgPT
# problem statement: This project/architecture focus on answerring user asked queries from the question bank
#usecase-chatbots

In [5]:
#chatbot-->Every ecommerce platform/health care/educational/department-They are integrating these chatbots nowadays
# Reason? Answering user related queries in less time with best answers by flagging irrevelant topics


In [6]:
#Flow

# pdfmaterial->Extract_text_pdf(perpage)->split_text_into_chunks->Generate_embeddings->

#A neural network is a machine learning model inspired by biological neurons-->Embeddings(List of vectors)-->
#store_embeddings-->FAISS-->Top-k-similarity search-->contextual response

In [7]:
pip install -q pypdf google-generativeai faiss-cpu numpy

Note: you may need to restart the kernel to use updated packages.


In [8]:
#projectsetup

# step-1:import packages

import os
import getpass
import numpy as np
import faiss
from pypdf import PdfReader
import google.generativeai as genai

C:\Users\yadda\AppData\Local\Temp\ipykernel_5856\1235077688.py:10: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [9]:
#step-2:gemini api configuration

GOOGLE_API_KEY='AQ.Ab8RN6IvuDoPM2WhW-hgOIODhxyQ1HDu3A8MgKZoqk5-LIXZpg'
genai.configure(api_key=GOOGLE_API_KEY)

GEN_MODEL_NAME='gemini-2.5-flash'
EMBED_MODEL_NAME='models/gemini-embedding-001'

gen_model=genai.GenerativeModel(GEN_MODEL_NAME)

In [23]:
#STEP-3: PDF LOADER(PURPOSE: Extracting text from PDF)

#Responsible for extracting text(single pdf,multiple pdfs--We gonna parse(scraping) each page)
def load_pdf(file_path):
    """
    Extracts text from a PDF, page by page.
    Returns: list of dicts -> [{"source": filename, "page": 1, "text": "..."}, ...]
    """
    reader = PdfReader(file_path)
    source_name = os.path.basename(file_path)
    pages_data = []

    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = text.strip()
        if text:
            pages_data.append({
                "source": source_name,
                "page": i,
                "text": text
            })

    print(f"Loaded '{source_name}' -> {len(pages_data)} pages with text.")
    return pages_data


def load_multiple_pdfs(file_paths):
    """Loads and combines page data from multiple PDFs."""
    all_pages = []
    for path in file_paths:
        all_pages.extend(load_pdf(path))
    return all_pages

In [24]:
#Module-2: chunking mechanism

#The text we have received from module-1--->chunking(breaking down complex text into small parts)

def chunk_pages(pages_data, chunk_size=150, overlap=30):
    """
    Splits page-level text into overlapping chunks.
    Returns: list of dicts -> [{"id", "text", "source", "page"}, ...]
    """
    chunks = []
    chunk_id = 0

    for page in pages_data:
        words = page["text"].split()
        start = 0

        while start < len(words):
            end = start + chunk_size
            chunk_words = words[start:end]
            chunk_text = " ".join(chunk_words)

            chunks.append({
                "id": chunk_id,
                "text": chunk_text,
                "source": page["source"],
                "page": page["page"]
            })
            chunk_id += 1

            if end >= len(words):
                break
            start = end - overlap  # slide window back by `overlap` words

    print(f"Created {len(chunks)} chunks.")
    return chunks

In [25]:
#module-3: Text Embeddings(converting text to vector)

# artificial intelligence(AI) can be tackled by any junior level software engineer,with just 2 things--Accountability--->[0.45.0.23,]
# Judge the output,You are not gonna lose job because of AI, YOu gonna lose if you blindly trust the output of AI

def embed_texts(texts, task_type="retrieval_document", batch_size=1):
    """
    Embeds a list of texts using Gemini's embedding model.
    task_type:
        - "retrieval_document" for chunks going INTO the index
        - "retrieval_query"    for the user's question at search time
    Returns: numpy array of shape (len(texts), embedding_dim)
    """
    all_embeddings = []
    for text in texts:
        result = genai.embed_content(
            model=EMBED_MODEL_NAME,
            content=text,
            task_type=task_type
        )
        all_embeddings.append(result["embedding"])

    return np.array(all_embeddings, dtype="float32")

In [26]:
#module-4: Vector size(FAISS)

# question bank(10 concepts related to AI)

# user: What is AI??
# model: It have go through all these 10AI concepts

# Each concept will be assigend a score->Top 3 will be picked up, And the model will take care of the rest(blend,share optimal solution)

def build_vector_index(chunks):
    """
    Embeds all chunks and builds a FAISS index over them.
    Returns: (faiss_index, chunks) — chunks order matches index order.
    """
    texts = [c["text"] for c in chunks]
    embeddings = embed_texts(texts, task_type="retrieval_document")

    # normalize so inner product == cosine similarity
    faiss.normalize_L2(embeddings)

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    print(f"Vector index built: {index.ntotal} chunks, dimension {dim}.")
    return index

In [27]:
#module-5: Retriever(Top-k similarity search)

def retrieve_relevant_chunks(question, index, chunks, top_k=4):
    """
    Returns the top_k most relevant chunks for a question, each with
    a similarity score attached.
    """
    query_embedding = embed_texts([question], task_type="retrieval_query")
    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        chunk = chunks[idx].copy()
        chunk["score"] = float(score)
        results.append(chunk)

    return results

In [28]:
#module-6: Prompt builder

#we are instructing what model have to do, what precautions need to taken, And how to hadnle edge cases(ai,ml,dbms...)
#ipl,,,our model should flag and reject that user input

def build_prompt(question, retrieved_chunks, chat_history=None):
    context_blocks = []
    for c in retrieved_chunks:
        context_blocks.append(
            f"[Source: {c['source']}, Page: {c['page']}]\n{c['text']}"
        )
    context = "\n\n---\n\n".join(context_blocks)

    history_text = ""
    if chat_history:
        for turn in chat_history[-6:]:  # keep last 3 exchanges
            history_text += f"{turn['role'].capitalize()}: {turn['content']}\n"

    prompt = f"""You are CampusGPT, an AI learning assistant for students.
Answer ONLY using the context below, which comes from the student's own
uploaded documents.

If the answer is not present in the context, reply exactly with:
"I couldn't find that information in the uploaded PDF(s)."

Cite the source and page for any fact you use, like this: (Source: file.pdf, Page: 3)

Previous conversation (if relevant):
{history_text}

Context:
{context}

Question:
{question}
"""
    return prompt

In [29]:
#module-7: Answer generation

def generate_answer(prompt):
    response=gen_model.generate_content(prompt)
    return response.text

In [30]:
#module-8:conversation memory

chat_history=[]


In [31]:
#module-9:full pipeline(UI)

def ask(question, index, chunks, top_k=4, verbose=True):
    retrieved = retrieve_relevant_chunks(question, index, chunks, top_k=top_k)

    if not retrieved:
        answer = "I couldn't find that information in the uploaded PDF(s)."
    else:
        prompt = build_prompt(question, retrieved, chat_history)
        answer = generate_answer(prompt)

    chat_history.append({"role": "user", "content": question})
    chat_history.append({"role": "assistant", "content": answer})

    if verbose:
        print("Q:", question)
        print("\nA:", answer)
        if retrieved:
            print("\nSources used:")
            for c in retrieved:
                print(f"  - {c['source']} · page {c['page']} · similarity {c['score']:.3f}")

    return answer, retrieved

In [32]:
#module-10: Testing
pdf_files=['AI_data.pdf']

In [33]:
pages_data=load_multiple_pdfs(pdf_files)  #return scraped data
chunks=chunk_pages(pages_data,chunk_size=150,overlap=30)
vector_index=build_vector_index(chunks)

Loaded 'AI_data.pdf' -> 3 pages with text.
Created 6 chunks.
Vector index built: 6 chunks, dimension 3072.


In [35]:
#Final step-chatting with chatbot

ask('what is ML',vector_index,chunks)

Q: what is ML

A: Machine Learning (ML) is a subset of AI in which systems learn patterns from data instead of relying only on explicitly written rules. (Source: AI_data.pdf, Page: 1)

Sources used:
  - AI_data.pdf · page 1 · similarity 0.695
  - AI_data.pdf · page 1 · similarity 0.693
  - AI_data.pdf · page 3 · similarity 0.675
  - AI_data.pdf · page 2 · similarity 0.663


('Machine Learning (ML) is a subset of AI in which systems learn patterns from data instead of relying only on explicitly written rules. (Source: AI_data.pdf, Page: 1)',
 [{'id': 1,
   'text': 'neural networks with multiple layers to learn complex patterns from large amounts of data. AI → Machine Learning → Deep Learning 3. Types of Machine Learning \x7f Supervised Learning: The model learns from labeled examples. Examples include spam detection, house-price prediction, and student-placement prediction. \x7f Unsupervised Learning: The model discovers patterns or groups in unlabeled data. Clustering customers into different groups is a common example. \x7f Reinforcement Learning: An agent learns by interacting with an environment and receiving rewards or penalties. It is used in robotics, games, and decision-making systems. 4. Neural Networks A neural network is a machine learning model inspired by biological neurons. It contains interconnected nodes organized into layers. During traini